# Text Processing & Find

## What's covered

- **The Unix philosophy** — small tools that do one thing, composed with pipes
- **`grep`** — searching for patterns in text. The most-used text tool on Linux.
- **Regular expressions** — the brief tour: anchors, character classes, repetition, alternation. Just enough to be productive.
- **`sed`** — the stream editor. Focus on the `s/old/new/` substitution form, which is 95% of real-world use.
- **`awk`** — the pattern-action language. Fields (`$1`, `$NF`), simple programs, and when to reach for it.
- **The supporting cast** — `cut`, `sort`, `uniq`, `tr`, `wc`, `head`, `tail` — small sharp tools you'll chain constantly.
- **The "top-N by frequency" idiom** — `sort | uniq -c | sort -rn | head` — memorise this one.
- **`find`** — searching the filesystem by name, type, size, modification time. The `-exec` action for running commands on results.
- **Archives and compression** — `tar`, `gzip`, `bzip2`, `xz`, `zstd`. The "tar pipe" idiom for moving directory trees around without intermediate files.

This notebook assumes notebooks 01-03: you can navigate, edit files, redirect output, and use pipes. Text processing is where Linux shines — the tools in this notebook compose with each other and with pipes (from notebook 02) to solve problems that would take pages of Python or Java to write.

## The Unix philosophy

Most operating systems give you a few large programs that try to do everything. Microsoft Word does word-processing and grammar-checking and mail-merging and spreadsheet-embedding all in one. Linux takes the opposite approach:

> **Write programs that do one thing and do it well. Write programs to work together. Write programs to handle text streams, because that is a universal interface.**

That's the Unix philosophy, articulated by Doug McIlroy in 1978. Every tool in this notebook is a small program with a narrow job:

- **`grep`** filters lines by pattern.
- **`sed`** rewrites lines.
- **`awk`** processes lines as records of fields.
- **`cut`** extracts columns.
- **`sort`** sorts lines.
- **`uniq`** collapses adjacent duplicates.
- **`tr`** translates characters.
- **`wc`** counts.
- **`find`** locates files.
- **`tar`** bundles files.

None of these is impressive on its own. But chained together with pipes, they let you do extraordinarily complex text manipulation in a single line. The "Linux pipeline" is one of the most powerful and underrated programming environments ever built — and you have everything you need to write them after notebooks 01-03.

This notebook teaches the tools. Notebook 10 returns to pipelines and shell scripting in full.

## `grep` — searching for text

`grep` ("global regular expression print") reads input line by line, prints the lines that match a pattern, and discards the rest. It's the most-typed command in Linux after `ls`.

The basic shape:

```
grep PATTERN file              # search a file
grep PATTERN file1 file2 ...   # search multiple files
some-command | grep PATTERN    # search the output of another command
```

The pattern can be a simple string or a regular expression (covered in the next section). For now, treat it as a string and quote it to be safe:

```bash
grep "error" /var/log/syslog
grep "root" /etc/passwd
ps -ef | grep "bash"
```

The flags you'll use constantly:

| Flag | Effect |
|---|---|
| **`-i`** | case-insensitive match |
| **`-v`** | invert: print lines that *don't* match |
| **`-n`** | prefix each output line with its line number |
| **`-c`** | print only the count of matching lines |
| **`-l`** | print only filenames that contain a match (useful with `-r`) |
| **`-L`** | print only filenames that don't contain a match |
| **`-r`**, **`-R`** | recurse into directories |
| **`-w`** | match whole words only |
| **`-o`** | print only the matching part, not the whole line |
| **`-A 3`** | print 3 lines **after** each match |
| **`-B 3`** | print 3 lines **before** each match |
| **`-C 3`** | print 3 lines of **context** (both before and after) |
| **`-F`** | treat pattern as fixed string, not regex (faster, no escaping needed) |
| **`-E`** | use **extended** regex (you almost always want this — see next section) |

If you find yourself searching code, install **`ripgrep`** (the `rg` command) — it's a modern grep, dramatically faster, respects `.gitignore`, and has saner regex defaults.

In [ ]:
!grep root /etc/passwd
!grep -c bash /etc/passwd
!grep -i "ERROR" /var/log/syslog 2>/dev/null | head -3
!grep -n root /etc/passwd
!grep -v "^#" /etc/ssh/sshd_config 2>/dev/null | grep -v "^$" | head -10

## Regular expressions — the brief tour

A **regular expression** (regex) is a tiny language for describing string patterns. `grep`, `sed`, and `awk` all use them. The full language is large; the parts you actually need for LFCS-level work are small.

**Two flavours** (this is the first thing to know):

- **Basic regex (BRE)** — the default in `grep` and `sed`. Most metacharacters need backslash-escaping: `\(`, `\)`, `\+`, `\?`, `\|`.
- **Extended regex (ERE)** — enabled with `grep -E`, `sed -E`, or `awk` (which uses ERE by default). Metacharacters work without escaping.

**Use `-E` by default.** It's what people mean by "regex" today. Examples below use ERE.

The pieces you'll meet most:

| Pattern | Matches |
|---|---|
| **`.`** | any single character (except newline) |
| **`^`** | start of line (anchor) |
| **`$`** | end of line (anchor) |
| **`*`** | zero or more of the previous character |
| **`+`** | one or more of the previous character |
| **`?`** | zero or one (i.e. optional) |
| **`{3}`** | exactly 3 |
| **`{2,5}`** | between 2 and 5 |
| **`[abc]`** | any one of `a`, `b`, `c` (character class) |
| **`[a-z]`** | any lowercase letter (range) |
| **`[^abc]`** | anything **except** `a`, `b`, `c` (negated class) |
| **`(foo\|bar)`** | `foo` or `bar` (alternation — needs `-E` or escape in BRE) |
| **`\d`**, **`\w`** | digit, word character (**PCRE only** — use `grep -P`, not portable) |

Useful character classes (POSIX, portable):

- `[[:digit:]]` — digits 0-9
- `[[:alpha:]]` — letters
- `[[:alnum:]]` — letters and digits
- `[[:space:]]` — whitespace

Two practical patterns to memorise:

- **`^pattern`** — only lines that *start* with `pattern`.
- **`pattern$`** — only lines that *end* with `pattern`.

Don't try to absorb the whole regex language at once. Start with the four anchors (`^`, `$`), character classes (`[abc]`), and repetition (`*`, `+`). Those four building blocks cover most real-world greps.

In [ ]:
!grep -E "^root" /etc/passwd
!grep -E "bash$" /etc/passwd
!grep -E "^[a-z]+:" /etc/passwd | head -5
!grep -E "[0-9]+\.[0-9]+" /etc/hosts 2>/dev/null
!echo "phone: 555-1234, code: 12345" | grep -oE "[0-9]{3,5}"

## `sed` — the stream editor

`sed` reads input line by line, applies an editing script to each line, and writes to stdout. The vast majority of real-world `sed` use is **one command**: `s/old/new/` for substitution. Learn that and you've covered 95% of what you'll need.

The basic shape:

```
sed 's/PATTERN/REPLACEMENT/FLAGS' file
```

Without flags, only the **first** occurrence on each line is replaced:

```bash
echo "one two two two" | sed 's/two/TWO/'        # one TWO two two
echo "one two two two" | sed 's/two/TWO/g'       # one TWO TWO TWO — g for global
echo "Hello WORLD" | sed 's/world/Linux/i'       # case-insensitive
```

Like `grep`, **use `-E` for extended regex** so you don't have to backslash-escape `+`, `?`, `|`, `()`, `{}`:

```bash
echo "version 1.2.3" | sed -E 's/[0-9]+/X/g'     # version X.X.X
echo "Hello World" | sed -E 's/(Hello) (World)/\2 \1/'  # World Hello
```

Notice `\1` and `\2` — those are **back-references** to capture groups (the parts in `()`). Group 1 is the first parentheses, group 2 the second, and so on. This is how you reorder or restructure text.

Two more `sed` features worth knowing:

- **Choose a different delimiter** when the pattern contains `/`. Slashes inside the pattern have to be escaped, which gets ugly with file paths. Pick `|`, `#`, or anything else:

```bash
sed 's|/usr/bin|/opt/bin|g' file
```

- **In-place editing** with `-i`. Modifies the file directly instead of printing to stdout. GNU `sed` (Linux) wants `-i`; BSD `sed` (macOS) wants `-i ''` with an empty argument. For portability, write to a temp file and rename instead.

```bash
sed -i 's/old/new/g' file.txt      # GNU sed (Linux): edit file in place
sed -i.bak 's/old/new/g' file.txt  # ...and keep a .bak backup
```

When `sed` starts feeling clunky (multiple substitutions, conditional logic, anything stateful), it's time to reach for `awk` or a real scripting language. Use `sed` for what it's good at: simple substitutions in a pipeline.

In [ ]:
!echo "hello world" | sed 's/world/Linux/'
!echo "one two two" | sed 's/two/TWO/g'
!echo "name=alice,age=30,city=glasgow" | sed -E 's/,/\n/g'
!echo "  trim me   " | sed -E 's/^[[:space:]]+|[[:space:]]+$//g' | cat -A
!echo "v1.2.3" | sed -E 's/v([0-9]+)\.([0-9]+)\.([0-9]+)/major=\1 minor=\2 patch=\3/'

## `awk` — the pattern-action language

`awk` is a small programming language for record-oriented text. Mental model: for each input line, split into fields, then run a sequence of `pattern { action }` rules.

The defaults are what make it powerful:

- Input is split into **records** — one per line by default.
- Each record is split into **fields** — on whitespace by default.
- **`$1`** is the first field, **`$2`** the second, **`$NF`** the last field (where `NF` = number of fields).
- **`$0`** is the whole line.
- **`NR`** is the record number (line counter).

The simplest possible awk programs:

```bash
awk '{print $1}'                    # print first field of every line
awk '{print $NF}'                   # print last field
awk '{print $1, $3}'                # print first and third, space-separated
awk 'NR==1'                         # print only line 1
awk 'NR > 5'                        # print from line 6 onwards
awk 'length($0) > 80'               # lines longer than 80 chars
```

Patterns can be regular expressions (between `/.../`) — match-and-print:

```bash
awk '/error/' /var/log/syslog       # like grep error
awk '!/^#/' config.conf             # like grep -v ^#
```

Use **`-F`** to change the field separator. The classic example is `/etc/passwd`, which uses `:`:

```bash
awk -F: '{print $1}' /etc/passwd            # usernames
awk -F: '{print $1, $7}' /etc/passwd        # username and shell
awk -F: '$3 >= 1000 {print $1}' /etc/passwd # users with UID >= 1000 (real users)
```

**`BEGIN { ... }` and `END { ... }`** blocks run before any input and after all input. With **associative arrays** (`count[$1]++`), you have everything you need for ad-hoc data aggregation:

```bash
awk -F: '{shells[$7]++} END {for (s in shells) print shells[s], s}' /etc/passwd
```

That one line: for each line in `/etc/passwd`, take the 7th field (shell), count occurrences, then at the end print each shell with its count.

**When to reach for `awk` over `sed`:** anything involving fields, arithmetic, accumulation, or per-record state. `sed` for simple substitution; `awk` for anything that has columns.

In [ ]:
!awk -F: '{print $1}' /etc/passwd | head -5
!awk -F: '{print $1, $7}' /etc/passwd | head -5
!awk -F: '$3 >= 1000 {print $1}' /etc/passwd
!awk 'NR <= 3' /etc/hostname /etc/hosts 2>/dev/null
!awk -F: '{shells[$7]++} END {for (s in shells) print shells[s], s}' /etc/passwd | sort -rn

## The supporting cast — `cut`, `sort`, `uniq`

Three small tools that compose with `grep`/`sed`/`awk` to handle most text wrangling.

**`cut`** — extract columns. Two main modes:

- By **delimiter**: `cut -d DELIM -f FIELDS`. The most common use.
- By **character position**: `cut -c POSITIONS`. Less common but useful for fixed-width data.

```bash
cut -d: -f1 /etc/passwd                  # first field, colon-separated
cut -d: -f1,3,7 /etc/passwd              # fields 1, 3, and 7
cut -d, -f2- file.csv                    # field 2 to the end
cut -c1-5 file.txt                       # characters 1 through 5
```

`cut` is simpler than `awk` when you don't need arithmetic — and noticeably faster on huge files.

**`sort`** — sort lines. Defaults to ASCII lexicographic order, which means `10` sorts before `2`. The flags you'll use:

| Flag | Effect |
|---|---|
| **`-n`** | numeric sort (so `2` comes before `10`) |
| **`-r`** | reverse |
| **`-u`** | unique — collapse adjacent duplicates inline |
| **`-k N`** | sort by field N |
| **`-t DELIM`** | use DELIM as field separator (for `-k`) |
| **`-h`** | human-numeric (handles `1K`, `2M`, `3G`) |
| **`-V`** | version sort (`1.10` after `1.2`) |

```bash
sort file.txt                            # ASCII sort
sort -n numbers.txt                      # numeric sort
sort -t: -k3 -n /etc/passwd              # sort by UID
sort -rn -k2 data.txt | head -10         # top 10 by field 2
```

**`uniq`** — collapse **adjacent** duplicate lines. The "adjacent" is the crucial bit: `uniq` only sees consecutive runs. **Almost always preceded by `sort`** to make duplicates adjacent.

```bash
sort file.txt | uniq                     # dedupe (same as sort -u)
sort file.txt | uniq -c                  # prefix each line with its count
sort file.txt | uniq -d                  # show only duplicated lines
sort file.txt | uniq -u                  # show only unique lines (no duplicates)
```

In [ ]:
!cut -d: -f1 /etc/passwd | head -5
!cut -d: -f1,3,7 /etc/passwd | head -5
!sort -t: -k3 -n /etc/passwd | head -5
!awk -F: '{print $7}' /etc/passwd | sort | uniq -c

## More supporting cast — `tr`, `wc`, `head`, `tail`

**`tr`** ("translate") — replace or delete **individual characters**, character-class style. Not a full search-and-replace (use `sed` for that); `tr` operates one character at a time.

```bash
echo "Hello World" | tr 'a-z' 'A-Z'      # uppercase
echo "Hello, World!" | tr -d ',!'        # delete commas and exclamation marks
cat file.txt | tr -s ' '                 # squeeze consecutive spaces into one
cat file.txt | tr '\r' ''                # strip Windows carriage returns
```

`tr` only reads from stdin — it doesn't take a filename argument. You always pipe into it.

**`wc`** ("word count") — count lines, words, bytes:

| Flag | Counts |
|---|---|
| **`-l`** | lines (the one you'll use 90% of the time) |
| **`-w`** | words |
| **`-c`** | bytes |
| **`-m`** | characters (different from bytes in UTF-8) |

```bash
wc -l /etc/passwd                        # how many lines (users)
ls /etc | wc -l                          # how many entries in /etc
```

**`head`** and **`tail`** — first/last N lines. Default is 10; override with `-n N`:

```bash
head -20 file.txt                        # first 20 lines
tail -20 file.txt                        # last 20 lines
tail -n +5 file.txt                      # from line 5 onwards (note the +)
tail -f /var/log/syslog                  # follow mode — keep printing new lines
```

`tail -f` is the standard way to watch a log file. `Ctrl-C` to exit.

You've now met the small-tool toolkit. Time to combine them.

In [ ]:
!echo "Hello, World!" | tr 'a-z' 'A-Z'
!echo "extra   whitespace   here" | tr -s ' '
!wc -l /etc/passwd /etc/group /etc/hosts
!head -3 /etc/passwd
!tail -3 /etc/passwd

## The "top-N by frequency" idiom

A pipeline pattern that comes up constantly: given a stream of items, find the most common ones.

```
... | sort | uniq -c | sort -rn | head -N
```

Read left to right:

1. **`sort`** — get duplicate items next to each other (so `uniq` can see them).
2. **`uniq -c`** — collapse adjacent duplicates, prefixing each with a count.
3. **`sort -rn`** — sort by that count, numerically and in reverse (highest first).
4. **`head -N`** — keep only the top N.

This idiom answers questions like:

- "What are the most common shells used by users on this system?"
- "Which IP addresses appear most in this access log?"
- "Which directories under `/etc` have the most files?"
- "What are the top error messages in syslog?"

A worked example. Here's how to find the most popular shells on the system:

```bash
awk -F: '{print $7}' /etc/passwd | sort | uniq -c | sort -rn | head
```

Walking through: `awk -F: '{print $7}'` extracts the shell field from each line of `/etc/passwd`. The result is piped to `sort`, then `uniq -c` collapses runs with counts, then `sort -rn` sorts by count descending, then `head` takes the top entries.

You'll write variations of this pipeline forever. Memorise the shape.

In [ ]:
!awk -F: '{print $7}' /etc/passwd | sort | uniq -c | sort -rn | head
!ls /etc 2>/dev/null | awk -F. '{print $NF}' | sort | uniq -c | sort -rn | head -10

## `find` — searching the filesystem

`grep` searches *inside* files. `find` searches *for* files — by name, type, size, modification time, ownership, or any combination. It's slower than its rivals (`locate`, `fd`, `mlocate`) because it walks the actual tree on every run, but it's everywhere — even on the most minimal systems.

The basic shape is **starting paths**, then a **chain of predicates**:

```
find PATH... [predicates]
```

If you don't give any predicates, `find` lists everything under the starting paths. Useful occasionally:

```bash
find /etc                # everything under /etc (huge output, redirect or pipe)
find . -type f | wc -l   # count all regular files under current dir
```

The most useful predicates:

| Predicate | Matches |
|---|---|
| **`-name PATTERN`** | files whose basename matches glob `PATTERN`. **Quote the pattern!** |
| **`-iname PATTERN`** | case-insensitive `-name` |
| **`-type f`** | regular files |
| **`-type d`** | directories |
| **`-type l`** | symbolic links |
| **`-size +100M`** | larger than 100 megabytes (`+` for more than, `-` for less than) |
| **`-size -1k`** | less than 1 kilobyte |
| **`-mtime -7`** | modified within the last 7 days (`-` = less than N days ago) |
| **`-mtime +30`** | modified more than 30 days ago |
| **`-mmin -10`** | modified within the last 10 *minutes* |
| **`-user NAME`** | owned by user `NAME` |
| **`-group NAME`** | owned by group `NAME` |
| **`-perm 644`** | exactly mode 644 |
| **`-perm -u+x`** | user-execute bit is set (the `-` prefix means "at least these bits") |
| **`-empty`** | empty file or directory |
| **`-maxdepth N`** | don't recurse deeper than N levels |

Predicates combine with **`-a`** (and, the default — just put them next to each other), **`-o`** (or), and **`!`** (not). Group with `\( ... \)`.

A useful escape hatch is **`-prune`** — used to skip directories you don't want to descend into, like `node_modules` or `.git`.

In [ ]:
!find /etc -maxdepth 1 -name "*.conf" 2>/dev/null
!find /var/log -maxdepth 2 -type f -size +1M 2>/dev/null | head -5
!find /etc -maxdepth 1 -type l 2>/dev/null | head -5
!find /tmp -maxdepth 1 -mmin -60 2>/dev/null | head -5

## `find -exec` — acting on results

`find` becomes truly powerful when paired with **`-exec`**, which runs a command on each match (or batches of matches). Two forms:

**`-exec CMD {} \;`** — runs CMD once per match. The `{}` is replaced with the filename; `\;` ends the command (the backslash protects the semicolon from the shell). Slow for many matches because it forks one process per file.

**`-exec CMD {} +`** — runs CMD once, appending all matches as arguments. Much faster. **Prefer this form** unless you specifically need per-file execution.

```bash
find . -name "*.tmp" -exec rm {} +                # remove all .tmp files (one rm call)
find /var/log -type f -name "*.log" -exec wc -l {} +    # count lines in all logs
find . -type d -exec chmod 755 {} +               # set 755 on every directory
find . -type f -exec chmod 644 {} +               # set 644 on every file
```

That last pair is the **correct way** to do "755 for dirs, 644 for files" recursively. Doing a flat `chmod -R 755` would incorrectly make all regular files executable (you saw this gotcha in notebook 03).

`find` also has a built-in **`-delete`** action, faster and safer than `-exec rm`:

```bash
find /tmp -name "*.tmp" -mtime +7 -delete         # delete temp files older than a week
find . -type d -empty -delete                     # remove empty directories
```

**Be careful with deletion.** Always run the `find` without `-delete` first to inspect what would be removed.

One more useful trick: **`xargs`**. It reads input from stdin and turns it into command arguments. Roughly equivalent to `-exec +` but works with any command that produces a list:

```bash
find . -name "*.log" | xargs gzip                 # gzip every .log file
find . -name "*.log" -print0 | xargs -0 gzip      # safer: handle filenames with spaces
```

The `-print0` + `xargs -0` pair uses null separators instead of newlines — important when filenames might contain spaces or newlines.

In [ ]:
!find /etc -maxdepth 1 -type f -name "*.conf" -exec wc -l {} + 2>/dev/null | tail -10
!find /tmp -maxdepth 1 -name "test-*" -mmin -1 2>/dev/null

## Archives and compression — `tar` plus the lineup

Two separate things to keep clear:

- **Archive** — bundle many files into one stream. Linux uses `tar` ("tape archive").
- **Compress** — make a stream smaller. Several algorithms; pick by trade-off.

**`tar` has three operations** you'll use:

```bash
tar -czf archive.tar.gz dir/      # Create gZipped: c = create, z = gzip, f = file
tar -xzf archive.tar.gz           # eXtract gZipped: x = extract, z = gzip, f = file
tar -tzf archive.tar.gz           # lisT (peek without extracting)
```

The compression flag matches the algorithm:

| Flag | Algorithm | Filename convention |
|---|---|---|
| **`-z`** | gzip | `.tar.gz` or `.tgz` |
| **`-j`** | bzip2 | `.tar.bz2` |
| **`-J`** (capital) | xz / lzma | `.tar.xz` |
| **`--zstd`** | zstd | `.tar.zst` |

Modern `tar` **autodetects on extract** — `tar -xf archive.tar.gz` works for any compression. You only need to specify on creation.

**Choosing a compression algorithm:**

| Algorithm | Speed | Ratio | When to use |
|---|---|---|---|
| **gzip** (`.gz`) | fast | mediocre | maximum compatibility — default for most things |
| **bzip2** (`.bz2`) | slow | better than gzip | mostly historical now |
| **xz** (`.xz`) | very slow to compress, fast to decompress | excellent | distribution packages, archival |
| **zstd** (`.zst`) | very fast both ways | very good | the modern default for new work |

For LFCS-level work, just know `gzip` and `xz`. **`zstd`** is increasingly common (it's the default in modern systemd journal, Btrfs compression, and recent distros).

Some useful flags:

- **`-v`** — verbose (print each file as it's processed)
- **`-C DIR`** — change to DIR before doing anything (controls where files are extracted/sourced)
- **`-p`** — preserve permissions on extract (default for root, not for normal users)

In [ ]:
!mkdir -p /tmp/sample && echo "alpha" > /tmp/sample/a.txt && echo "beta" > /tmp/sample/b.txt
!tar -czvf /tmp/sample.tar.gz -C /tmp sample 2>&1
!tar -tzf /tmp/sample.tar.gz
!ls -lh /tmp/sample.tar.gz
!mkdir -p /tmp/restored && tar -xzf /tmp/sample.tar.gz -C /tmp/restored && ls /tmp/restored/sample
!rm -rf /tmp/sample /tmp/sample.tar.gz /tmp/restored

## The "tar pipe" idiom

A pattern worth knowing once you're comfortable with `tar`: transferring a directory tree from one place to another **without** creating an intermediate archive file.

```bash
tar -cf - source-dir/ | tar -xf - -C /destination/
```

The trick: when you use `-f -`, `tar` reads from or writes to **stdout/stdin** instead of a file. So you can pipe one `tar` directly into another. No archive file ever exists on disk; the bytes flow from one process to the other.

The classic use is **transferring over SSH** (covered in notebook 08):

```bash
tar -czf - source/ | ssh remote 'tar -xzf - -C /destination'
```

Compress on this side, stream over ssh, decompress on the other side. No intermediate file, no double disk I/O. You'll see this idiom constantly in operations work.

For everyday "copy this directory to that machine" jobs you'd reach for **`rsync`** (notebook 08), which is even better — it transfers only changed files and is restartable. But the `tar` pipe is universally available, even on the most minimal systems, and worth knowing.

For now, the takeaway: `-f -` means "use stdin/stdout instead of a file." That trick makes `tar` composable with other shell tools.

## What you've learned

A compact recap to keep next to your prompt:

- **The Unix philosophy** — small tools that do one thing, composed with pipes.
- **`grep`** — find lines matching a pattern. `-i` (case-insensitive), `-v` (invert), `-n` (line numbers), `-c` (count), `-r` (recurse), `-A/-B/-C` (context).
- **Regex** — start with `^` (start), `$` (end), `[abc]` (class), `*`/`+`/`?` (repetition), `()` (groups). Use **`-E`** for extended regex.
- **`sed 's/old/new/g'`** — the substitution form. Pick a delimiter that doesn't clash. `-E` for extended regex. `-i` for in-place editing (Linux only).
- **`awk`** — fields with `$1`, `$NF`, `$0`. `-F:` for field separator. `BEGIN { } / pattern { action } / END { }`. Associative arrays for counting.
- **`cut`** — extract columns by delimiter (`-d -f`) or position (`-c`). Faster than `awk` when you don't need its features.
- **`sort`** — `-n` numeric, `-r` reverse, `-u` unique, `-k N` by field, `-t` separator. **`sort -h`** for human-readable sizes (`1K`, `2M`).
- **`uniq`** — collapse **adjacent** duplicates. Always `sort | uniq -c` together. `-d` for duplicates only.
- **`tr`** — translate single characters. `'a-z' 'A-Z'` to uppercase. `-d` to delete. `-s` to squeeze repeats.
- **`wc`** — `-l` lines, `-w` words, `-c` bytes. **`wc -l`** is the one you'll type constantly.
- **`head -N` / `tail -N`** — first/last N lines. `tail -f` to follow.
- **The top-N idiom** — `... | sort | uniq -c | sort -rn | head`. Memorise the shape.
- **`find PATH predicates`** — `-name '*.log'`, `-type f`, `-size +1M`, `-mtime -7`, `-user X`, `-perm 644`. Use `-maxdepth N` to limit recursion. Combine with `-a`/`-o`/`!`.
- **`find -exec CMD {} +`** — batch action on results. **Prefer `+` over `\;`** for speed. **`-delete`** as a safer built-in for removal.
- **`tar -czf archive.tar.gz dir/`** to create, **`tar -xf archive.tar.gz`** to extract (autodetects compression). `-z` gzip, `-j` bzip2, `-J` xz, `--zstd` zstd. Use gzip for compatibility, zstd for new work.
- **The tar pipe** — `tar -cf - dir/ | something-else` — stream without an intermediate file.

**Practise before moving on.** Pick a real log file (e.g. `/var/log/syslog` or any web access log you have) and run pipelines on it: count lines, count by status code, find the top IPs, extract timestamps with `awk` or `cut`. Linux text-processing rewards play more than reading.

**Next:** notebook `05-processes-jobs-and-signals` introduces processes — what they are, how to view them with `ps`/`top`/`htop`, how to control them with `&`, `fg`, `bg`, `nohup`, and how to send them signals (`kill`, `SIGTERM`, `SIGKILL`).